In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv()

In [ ]:
memory_path = os.path.abspath("memory/memory.json")
memory_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-memory"], "env": {"MEMORY_FILE_PATH": memory_path}}

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=60) as server:
    memory_tools = await server.list_tools()

memory_tools

In [ ]:
instructions = "You use your entity tools as a persistent memory top store and recall informaqtion about your conversation"
request = "My name's Sage. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP Protocol"
model = "gemini-3.5-flash"

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent[any](name="agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent[any](name = "agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("converstaion"):
        result = await Runner.run(agent, "My name's Sage. What do you know about me?")
    display_markdown(result.final_output)

In [ ]:
tavily_params = {"command": "npx", "args": ["-y", "tavily-mcp@latest"], "env": {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY") }}

async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60) as server:
    tavily_tools = await server.list_tools()

tavily_tools

In [ ]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize it's outlook. For context, it is {datetime}"
model = "gemini-3.5-flash"
search_only = create_static_tool_filter(allowed_tool_names=["tavily_search"])

In [ ]:
async with MCPServerStdio(params=tavily_params, client_session_timeout_seconds=60, tool_filter=search_only) as mcp_server:
    agent = Agent[any](name="agent", instruction=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))